# Chicago Crime Data Pipeline

## 1. Schema Setup
Create the three-tier architecture: Bronze (raw data), Silver (cleaned/enriched), Gold (aggregated analytics)

## 2. Bronze Layer - Data Ingestion
Load raw data from Chicago Data Portal APIs into Bronze tables

## 3. Data Validation
Verify Bronze layer data quality and row counts

## 4. Silver Layer - Data Transformation Pipeline

Transformation sequence:
1. **Cast**: Convert string columns to proper data types
2. **Filter**: Remove records with missing critical fields
3. **Dedupe**: Remove duplicate crime_id records
4. **Enrich**: Add temporal features (month, day_of_week, hour_of_day)
5. **Join**: Add community area names from reference data

## 5. Data Quality Check
Verify final Silver table completeness - check null percentages for key columns

In [0]:
CREATE SCHEMA IF NOT EXISTS bronze;
CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS gold;

### Load Crime Records from Chicago Data Portal
Fetches 4 years of crime data (2019-2022) from the Chicago Open Data API in batches of 50K records per year. Writes raw data to `bronze.chicago_crimes_raw` Delta table.

### Load Community Area Reference Data
Fetches Chicago's 77 community area names and geographic boundaries. Writes to `bronze.community_areas_raw` for later enrichment joins.

### Validate Bronze Layer
Verifies both Bronze tables loaded successfully by checking row counts.

### Silver Transformation Step 1: Cast to Proper Types
Converts string columns to appropriate data types:
* `date` → timestamp
* `beat`, `district`, `ward`, `community_area`, `year` → int
* `latitude`, `longitude` → double
* `arrest`, `domestic` → boolean
* Standardizes crime_type to uppercase

### Silver Transformation Step 2: Filter for Data Quality
Removes records with missing critical fields:
* crime_id, crime_type, date must be present
* latitude, longitude, community_area must be present
* year must be between 2000-2025 (sanity check)

### Silver Transformation Step 3: Deduplicate Records
Uses ROW_NUMBER() to keep only the most recent record per crime_id. Partitions by crime_id and orders by date DESC to retain the latest version.

### Silver Transformation Step 4: Add Temporal Features
Extracts time-based features for analysis:
* `month` (1-12)
* `day_of_week` (1-7, where 1=Sunday)
* `hour_of_day` (0-23)
* `day_name` (Monday, Tuesday, etc.)

### Silver Transformation Step 5: Join Community Names
Left joins with Bronze community_areas_raw to add readable community area names. Converts community names to title case using INITCAP(). This is the final clean Silver table ready for analytics.

### Final Data Quality Validation
Checks null percentages for all critical columns to ensure data completeness before moving to Gold layer.

### Additional Validation Checks
Verifies no duplicate crime_ids remain and checks the date range coverage.

## 6. Gold Layer - Analytics Marts

### Create Crime-by-Area Analytics Table
Aggregates crime statistics by community area and year:
* Total crimes and arrests per area/year
* Arrest rate percentage
* Crime rankings (highest and lowest crime areas per year)

This table powers dashboards and reports on geographic crime patterns.

In [0]:
%python
# Import required libraries for API requests and data manipulation
import requests
import pandas as pd

# Function to fetch crime records from Chicago Data Portal API
def fetch_chicago_crimes(year, limit=50000):
    # API endpoint for Chicago crime data
    url = "https://data.cityofchicago.org/resource/ijzp-q8t2.json"
    all_records = []
    offset = 0  # Starting position for pagination

    # Loop to fetch data in batches (pagination)
    while True:
        # API query parameters: filter by year, set batch size and offset
        params = {
            "$where": f"year = {year}",      # Filter: only this year's records
            "$limit": limit,                  # Batch size: 50K records per request
            "$offset": offset,                # Starting position for this batch
            "$order": ":id"                   # Order by ID for consistent pagination
        }
        # Make HTTP GET request to API
        response = requests.get(url, params=params)
        batch = response.json()  # Parse JSON response
        
        # If no more records, exit loop
        if not batch:
            break
        
        # Add this batch to our collection
        all_records.extend(batch)
        offset += limit  # Move to next batch
        print(f"Year {year}: {len(all_records)} records fetched...")

    return pd.DataFrame(all_records)  # Convert to pandas DataFrame

# Define years to fetch (2019-2022)
years = [2019, 2020, 2021, 2022]
# Fetch data for each year and store in list
dfs = [fetch_chicago_crimes(y) for y in years]
# Combine all years into single DataFrame
df_raw = pd.concat(dfs, ignore_index=True)

# Convert pandas DataFrame to Spark DataFrame and write to Bronze layer
spark.createDataFrame(df_raw).write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze.chicago_crimes_raw")

print(f"Done. {len(df_raw):,} rows written to bronze.chicago_crimes_raw")

Year 2019: 50000 records fetched...
Year 2019: 100000 records fetched...
Year 2019: 150000 records fetched...
Year 2019: 200000 records fetched...
Year 2019: 250000 records fetched...
Year 2019: 261711 records fetched...
Year 2020: 50000 records fetched...
Year 2020: 100000 records fetched...
Year 2020: 150000 records fetched...
Year 2020: 200000 records fetched...
Year 2020: 212729 records fetched...
Year 2021: 50000 records fetched...
Year 2021: 100000 records fetched...
Year 2021: 150000 records fetched...
Year 2021: 200000 records fetched...
Year 2021: 209687 records fetched...
Year 2022: 50000 records fetched...
Year 2022: 100000 records fetched...
Year 2022: 150000 records fetched...
Year 2022: 200000 records fetched...
Year 2022: 240053 records fetched...
Done. 924,180 rows written to bronze.chicago_crimes_raw


In [0]:
%python
# Import required libraries
import requests, pandas as pd

# Fetch community area reference data from Chicago Data Portal
response = requests.get(
    "https://data.cityofchicago.org/resource/igwz-8jzy.json",  # API endpoint
    params={"$limit": 100}  # Request up to 100 records (77 areas exist)
)
# Parse JSON response into pandas DataFrame
df_areas = pd.DataFrame(response.json())

# Convert to Spark DataFrame and write to Bronze layer
spark.createDataFrame(df_areas).write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze.community_areas_raw")

print("Done. Community areas written to bronze.community_areas_raw")

Done. Community areas written to bronze.community_areas_raw


In [0]:
SELECT 'chicago_crimes_raw'  AS table_name, COUNT(*) AS row_count FROM bronze.chicago_crimes_raw
UNION ALL
SELECT 'community_areas_raw' AS table_name, COUNT(*) AS row_count FROM bronze.community_areas_raw;

table_name,row_count
chicago_crimes_raw,924180
community_areas_raw,77


In [0]:
SELECT * FROM bronze.chicago_crimes_raw LIMIT 5;

id,case_number,date,block,iucr,primary_type,description,location_description,arrest,domestic,beat,district,ward,community_area,fbi_code,x_coordinate,y_coordinate,year,updated_on,latitude,longitude,location
11900609,JC521530,2019-11-23T11:32:00.000,054XX W CHICAGO AVE,1812,NARCOTICS,POSS: CANNABIS MORE THAN 30GMS,PARKING LOT/GARAGE(NON.RESID.),true,false,1524,015,37,25,18,1139866,1904803,2019,2019-11-30T15:41:40.000,41.8948813,-87.761762327,"List({""address"": """", ""city"": """", ""state"": """", ""zip"": """"}, 41.8948813, -87.761762327)"
11900610,JC521776,2019-11-23T15:40:00.000,063XX S ALBANY AVE,0560,ASSAULT,SIMPLE,STREET,false,true,0823,008,17,66,08A,1156806,1862530,2019,2019-11-30T15:41:40.000,41.778552642,-87.700689417,"List({""address"": """", ""city"": """", ""state"": """", ""zip"": """"}, 41.778552642, -87.700689417)"
11900613,JC521738,2019-11-23T13:00:00.000,044XX S OAKENWALD AVE,0820,THEFT,$500 AND UNDER,STREET,false,false,0222,002,4,39,06,1185363,1875808,2019,2019-11-30T15:41:40.000,41.814364297,-87.595580588,"List({""address"": """", ""city"": """", ""state"": """", ""zip"": """"}, 41.814364297, -87.595580588)"
11900614,JC521780,2019-11-23T15:10:00.000,053XX S NATCHEZ AVE,0620,BURGLARY,UNLAWFUL ENTRY,RESIDENCE,false,false,0811,008,23,56,05,1133949,1868542,2019,2019-11-30T15:41:40.000,41.795481565,-87.784345558,"List({""address"": """", ""city"": """", ""state"": """", ""zip"": """"}, 41.795481565, -87.784345558)"
11900616,JC521763,2019-11-23T15:20:00.000,060XX N MAPLEWOOD AVE,0610,BURGLARY,FORCIBLE ENTRY,RESIDENCE-GARAGE,false,false,2413,024,40,2,05,1158231,1940088,2019,2019-11-30T15:41:40.000,41.991350083,-87.693344739,"List({""address"": """", ""city"": """", ""state"": """", ""zip"": """"}, 41.991350083, -87.693344739)"


In [0]:
SELECT * FROM bronze.community_areas_raw LIMIT 5;

the_geom area_numbe community area_num_1 shape_area shape_len List(List(List(List(List(-87.65455590025104, 41.99816614970252), List(-87.65573692713699, 41.99817520952651), List(-87.65577334679872, 41.99819571036795), List(-87.65580429073043, 41.998209147443845), List(-87.65584690994521, 41.998221336207955), List(-87.65591636009239, 41.99823236608657), List(-87.65600350940235, 41.99823617337557), List(-87.65607415529718, 41.99823505345825), List(-87.65636459403105, 41.99823244374171), List(-87.65646687610464, 41.99823152469403), List(-87.6565185691827, 41.99823130562602), List(-87.65690019335015, 41.99822968823771), List(-87.65706259350846, 41.998227539507), List(-87.65720871015589, 41.99822560582784), List(-87.65739786383061, 41.998223454213104), List(-87.65776516966109, 41.99821934188437), List(-87.65807577620295, 41.99821586355912), List(-87.65853312294092, 41.99820999112175), List(-87.65865677860859, 41.998208640890574), List(-87.65882397102085, 41.998206815852484), List(-87.65916099054422, 41.99820348731977), List(-87.65917658573863, 41.99820382780797), List(-87.6592145045504, 41.99820365972302), List(-87.65928561239271, 41.99820334458911), List(-87.65932696669353, 41.99820316151438), List(-87.659431548325, 41.99820269788534), List(-87.6595484635856, 41.998202179500105), List(-87.65972813841594, 41.99819881557301), List(-87.65975967221236, 41.998198225104524), List(-87.66005286300864, 41.99819273531639), List(-87.66027926705436, 41.99819214208317), List(-87.66046824670686, 41.99819004338788), List(-87.66052828703576, 41.998189376665074), List(-87.66054502344734, 41.99818919073466), List(-87.66055651906055, 41.99818906299788), List(-87.66063444116676, 41.998188197513954), List(-87.66101998384175, 41.99818391435505), List(-87.66119225299092, 41.99818163317218), List(-87.66149845098077, 41.9981775775187), List(-87.66183393608992, 41.998174323812556), List(-87.66201887581596, 41.99817296593227), List(-87.66241141764256, 41.99816886694295), List(-87.66261488159151, 41.9981667354851), List(-87.66284170904127, 41.99816438339006), List(-87.66305192209948, 41.998158231922865), List(-87.66317392274446, 41.99815976475031), List(-87.66352426867974, 41.99815558025525), List(-87.66361632387768, 41.99815467474306), List(-87.66401110698924, 41.99815079109377), List(-87.66425127522662, 41.99814870060441), List(-87.66444157366927, 41.99814704349601), List(-87.6647749123838, 41.9981438811979), List(-87.66480390557379, 41.99814363348658), List(-87.66521420870092, 41.9981401272432), List(-87.66540480717677, 41.998137074040464), List(-87.66550444691632, 41.998135477623634), List(-87.66574950028331, 41.998131551310415), List(-87.66601484695195, 41.99812907310495), List(-87.66614560634768, 41.99812785159919), List(-87.6662456809783, 41.998126916769635), List(-87.6666108077265, 41.99812352251155), List(-87.66674800815446, 41.99812208087359), List(-87.66692034451671, 41.99812027034438), List(-87.6673426114641, 41.9981161624018), List(-87.66765946793315, 41.99811306075006), List(-87.66783247244231, 41.99811070992512), List(-87.66796195087547, 41.99810895039277), List(-87.66821152344531, 41.99810555845549), List(-87.66837480917981, 41.99810386975227), List(-87.66855321960972, 41.99810174317024), List(-87.6690149735619, 41.998096352832256), List(-87.66917042975086, 41.998094075982706), List(-87.66938700048297, 41.998090903816205), List(-87.67009101253248, 41.9980815026772), List(-87.67037226408969, 41.99807774572292), List(-87.67047237514275, 41.99807679808804), List(-87.67059060013851, 41.998075678607634), List(-87.67069376678978, 41.99807470149537), List(-87.67085836365963, 41.99807314278571), List(-87.67128195549962, 41.99806535078155), List(-87.67131040507569, 41.99806483731891), List(-87.67175897247732, 41.99805674001302), List(-87.67199219323774, 41.99805149783883), List(-87.67215336543731, 41.99804787092475), List(-87.67227644643, 41.99804558093631), List(-87.67244846683681, 41.99804238050815), List(-87.67283143853332, 41.99803511728949), List(

In [0]:
-- Silver Step 1: Cast string columns to proper data types for analysis
CREATE OR REPLACE TABLE silver.crimes_cast
USING DELTA AS
SELECT
  -- Rename id to crime_id for clarity
  id                                        AS crime_id,
  case_number,
  -- Convert date string to timestamp for time-based analysis
  CAST(date AS TIMESTAMP)                   AS date,
  block,
  -- Rename iucr to iucr_code for clarity
  iucr                                      AS iucr_code,
  -- Standardize crime type: uppercase and trim whitespace
  UPPER(TRIM(primary_type))                 AS crime_type,
  -- Standardize description: uppercase and trim whitespace
  UPPER(TRIM(description))                  AS description,
  -- Rename for brevity
  location_description                      AS location_desc,
  -- Convert string true/false to boolean
  CAST(arrest AS BOOLEAN)                   AS arrest,
  CAST(domestic AS BOOLEAN)                 AS domestic,
  -- Convert numeric strings to integers
  CAST(beat AS INT)                         AS beat,
  CAST(district AS INT)                     AS district,
  CAST(ward AS INT)                         AS ward,
  CAST(community_area AS INT)               AS community_area,
  CAST(year AS INT)                         AS year,
  -- Convert coordinate strings to double precision floats
  CAST(latitude AS DOUBLE)                  AS latitude,
  CAST(longitude AS DOUBLE)                 AS longitude
FROM bronze.chicago_crimes_raw;

num_affected_rows,num_inserted_rows


In [0]:
SELECT 'crimes_cast' AS step, COUNT(*) AS row_count FROM silver.crimes_cast;

step,row_count
crimes_cast,924180


In [0]:
SELECT * FROM silver.crimes_cast LIMIT 5;

crime_id,case_number,date,block,iucr_code,crime_type,description,location_desc,arrest,domestic,beat,district,ward,community_area,year,latitude,longitude
11552577,JC100040,2019-01-01T00:31:00.000Z,032XX W LAWRENCE AVE,1310,CRIMINAL DAMAGE,TO PROPERTY,RESTAURANT,false,false,1713,17,33,14,2019,41.968444497,-87.709341738
11552587,JC100034,2019-01-01T00:05:00.000Z,006XX E 83RD PL,1310,CRIMINAL DAMAGE,TO PROPERTY,RESIDENCE,false,false,632,6,6,44,2019,41.742968219,-87.6084099
11552596,JC100045,2019-01-01T00:03:00.000Z,001XX W HURON ST,0430,BATTERY,AGGRAVATED: OTHER DANG WEAPON,HOTEL/MOTEL,false,false,1832,18,42,8,2019,41.894821547,-87.632133928
11552605,JC100030,2019-01-01T00:01:00.000Z,004XX N MONTICELLO AVE,143A,WEAPONS VIOLATION,UNLAWFUL POSS OF HANDGUN,ALLEY,true,false,1122,11,27,23,2019,41.889196391,-87.717403722
11552609,JC100028,2019-01-01T00:05:00.000Z,013XX S CENTRAL PARK AVE,0486,BATTERY,DOMESTIC BATTERY SIMPLE,APARTMENT,false,true,1011,10,24,29,2019,41.863419512,-87.71533469


In [0]:
SELECT * FROM silver.crimes_filtered LIMIT 5;

crime_id,case_number,date,block,iucr_code,crime_type,description,location_desc,arrest,domestic,beat,district,ward,community_area,year,latitude,longitude
11552577,JC100040,2019-01-01T00:31:00.000Z,032XX W LAWRENCE AVE,1310,CRIMINAL DAMAGE,TO PROPERTY,RESTAURANT,false,false,1713,17,33,14,2019,41.968444497,-87.709341738
11552587,JC100034,2019-01-01T00:05:00.000Z,006XX E 83RD PL,1310,CRIMINAL DAMAGE,TO PROPERTY,RESIDENCE,false,false,632,6,6,44,2019,41.742968219,-87.6084099
11552596,JC100045,2019-01-01T00:03:00.000Z,001XX W HURON ST,0430,BATTERY,AGGRAVATED: OTHER DANG WEAPON,HOTEL/MOTEL,false,false,1832,18,42,8,2019,41.894821547,-87.632133928
11552605,JC100030,2019-01-01T00:01:00.000Z,004XX N MONTICELLO AVE,143A,WEAPONS VIOLATION,UNLAWFUL POSS OF HANDGUN,ALLEY,true,false,1122,11,27,23,2019,41.889196391,-87.717403722
11552609,JC100028,2019-01-01T00:05:00.000Z,013XX S CENTRAL PARK AVE,0486,BATTERY,DOMESTIC BATTERY SIMPLE,APARTMENT,false,true,1011,10,24,29,2019,41.863419512,-87.71533469


In [0]:
SELECT * FROM silver.crimes_deduped LIMIT 5;

crime_id,case_number,date,block,iucr_code,crime_type,description,location_desc,arrest,domestic,beat,district,ward,community_area,year,latitude,longitude,row_num
11138622,JA495186,2021-05-21T00:01:00.000Z,019XX N PULASKI RD,1752,OFFENSE INVOLVING CHILDREN,AGGRAVATED CRIMINAL SEXUAL ABUSE BY FAMILY MEMBER,RESIDENCE,false,true,2534,25,35,20,2021,41.915798196,-87.726524412,1
11552577,JC100040,2019-01-01T00:31:00.000Z,032XX W LAWRENCE AVE,1310,CRIMINAL DAMAGE,TO PROPERTY,RESTAURANT,false,false,1713,17,33,14,2019,41.968444497,-87.709341738,1
11552587,JC100034,2019-01-01T00:05:00.000Z,006XX E 83RD PL,1310,CRIMINAL DAMAGE,TO PROPERTY,RESIDENCE,false,false,632,6,6,44,2019,41.742968219,-87.6084099,1
11552596,JC100045,2019-01-01T00:03:00.000Z,001XX W HURON ST,0430,BATTERY,AGGRAVATED: OTHER DANG WEAPON,HOTEL/MOTEL,false,false,1832,18,42,8,2019,41.894821547,-87.632133928,1
11552605,JC100030,2019-01-01T00:01:00.000Z,004XX N MONTICELLO AVE,143A,WEAPONS VIOLATION,UNLAWFUL POSS OF HANDGUN,ALLEY,true,false,1122,11,27,23,2019,41.889196391,-87.717403722,1


In [0]:
SELECT * FROM silver.crimes_enriched LIMIT 5;

crime_id,case_number,date,block,iucr_code,crime_type,description,location_desc,arrest,domestic,beat,district,ward,community_area,year,latitude,longitude,row_num,month,day_of_week,hour_of_day,day_name
11138622,JA495186,2021-05-21T00:01:00.000Z,019XX N PULASKI RD,1752,OFFENSE INVOLVING CHILDREN,AGGRAVATED CRIMINAL SEXUAL ABUSE BY FAMILY MEMBER,RESIDENCE,false,true,2534,25,35,20,2021,41.915798196,-87.726524412,1,5,6,0,Friday
11552577,JC100040,2019-01-01T00:31:00.000Z,032XX W LAWRENCE AVE,1310,CRIMINAL DAMAGE,TO PROPERTY,RESTAURANT,false,false,1713,17,33,14,2019,41.968444497,-87.709341738,1,1,3,0,Tuesday
11552587,JC100034,2019-01-01T00:05:00.000Z,006XX E 83RD PL,1310,CRIMINAL DAMAGE,TO PROPERTY,RESIDENCE,false,false,632,6,6,44,2019,41.742968219,-87.6084099,1,1,3,0,Tuesday
11552596,JC100045,2019-01-01T00:03:00.000Z,001XX W HURON ST,0430,BATTERY,AGGRAVATED: OTHER DANG WEAPON,HOTEL/MOTEL,false,false,1832,18,42,8,2019,41.894821547,-87.632133928,1,1,3,0,Tuesday
11552605,JC100030,2019-01-01T00:01:00.000Z,004XX N MONTICELLO AVE,143A,WEAPONS VIOLATION,UNLAWFUL POSS OF HANDGUN,ALLEY,true,false,1122,11,27,23,2019,41.889196391,-87.717403722,1,1,3,0,Tuesday


In [0]:
SELECT * FROM silver.chicago_crimes_clean LIMIT 5;

crime_id,case_number,date,block,iucr_code,crime_type,description,location_desc,arrest,domestic,beat,district,ward,community_area,year,latitude,longitude,row_num,month,day_of_week,hour_of_day,day_name,community_area_name
11138622,JA495186,2021-05-21T00:01:00.000Z,019XX N PULASKI RD,1752,OFFENSE INVOLVING CHILDREN,AGGRAVATED CRIMINAL SEXUAL ABUSE BY FAMILY MEMBER,RESIDENCE,false,true,2534,25,35,20,2021,41.915798196,-87.726524412,1,5,6,0,Friday,Hermosa
11552577,JC100040,2019-01-01T00:31:00.000Z,032XX W LAWRENCE AVE,1310,CRIMINAL DAMAGE,TO PROPERTY,RESTAURANT,false,false,1713,17,33,14,2019,41.968444497,-87.709341738,1,1,3,0,Tuesday,Albany Park
11552587,JC100034,2019-01-01T00:05:00.000Z,006XX E 83RD PL,1310,CRIMINAL DAMAGE,TO PROPERTY,RESIDENCE,false,false,632,6,6,44,2019,41.742968219,-87.6084099,1,1,3,0,Tuesday,Chatham
11552596,JC100045,2019-01-01T00:03:00.000Z,001XX W HURON ST,0430,BATTERY,AGGRAVATED: OTHER DANG WEAPON,HOTEL/MOTEL,false,false,1832,18,42,8,2019,41.894821547,-87.632133928,1,1,3,0,Tuesday,Near North Side
11552605,JC100030,2019-01-01T00:01:00.000Z,004XX N MONTICELLO AVE,143A,WEAPONS VIOLATION,UNLAWFUL POSS OF HANDGUN,ALLEY,true,false,1122,11,27,23,2019,41.889196391,-87.717403722,1,1,3,0,Tuesday,Humboldt Park


In [0]:
SELECT 'crimes_filtered' AS step, COUNT(*) AS row_count FROM silver.crimes_filtered;

step,row_count
crimes_filtered,904780


In [0]:
SELECT 'crimes_deduped' AS step, COUNT(*) AS row_count FROM silver.crimes_deduped;

step,row_count
crimes_deduped,904780


In [0]:
SELECT 'crimes_enriched' AS step, COUNT(*) AS row_count FROM silver.crimes_enriched;

step,row_count
crimes_enriched,904780


In [0]:
SELECT 'chicago_crimes_clean' AS step, COUNT(*) AS row_count FROM silver.chicago_crimes_clean;

step,row_count
chicago_crimes_clean,904780


In [0]:
-- Silver Step 2: Filter out records with missing critical fields
CREATE OR REPLACE TABLE silver.crimes_filtered
USING DELTA AS
SELECT *
FROM silver.crimes_cast
WHERE
  -- Ensure all critical identification fields are present
  crime_id        IS NOT NULL
  AND crime_type  IS NOT NULL
  AND date        IS NOT NULL
  -- Ensure geographic coordinates are present for mapping
  AND latitude    IS NOT NULL
  AND longitude   IS NOT NULL
  -- Ensure community area is present for area-based analysis
  AND community_area IS NOT NULL
  -- Sanity check: year should be realistic (2000-2025)
  AND year BETWEEN 2000 AND 2025;

num_affected_rows,num_inserted_rows


In [0]:
-- Silver Step 3: Remove duplicate crime records, keeping the most recent version
CREATE OR REPLACE TABLE silver.crimes_deduped
USING DELTA AS
SELECT *
FROM (
  SELECT *,
    -- Assign row number within each crime_id group
    ROW_NUMBER() OVER (
      PARTITION BY crime_id      -- Group by crime_id
      ORDER BY date DESC         -- Order by most recent date first
    ) AS row_num
  FROM silver.crimes_filtered
)
-- Keep only the first row (most recent) for each crime_id
WHERE row_num = 1;

num_affected_rows,num_inserted_rows


In [0]:
-- Silver Step 4: Add temporal features for time-based analysis
CREATE OR REPLACE TABLE silver.crimes_enriched
USING DELTA AS
SELECT
  *,  -- Keep all existing columns
  -- Extract month (1-12) for monthly trend analysis
  MONTH(date)                               AS month,
  -- Extract day of week (1=Sunday, 7=Saturday) for weekly patterns
  DAYOFWEEK(date)                           AS day_of_week,
  -- Extract hour (0-23) for hourly crime patterns
  HOUR(date)                                AS hour_of_day,
  -- Convert day number to readable day name
  CASE DAYOFWEEK(date)
    WHEN 1 THEN 'Sunday'
    WHEN 2 THEN 'Monday'
    WHEN 3 THEN 'Tuesday'
    WHEN 4 THEN 'Wednesday'
    WHEN 5 THEN 'Thursday'
    WHEN 6 THEN 'Friday'
    WHEN 7 THEN 'Saturday'
  END                                       AS day_name
FROM silver.crimes_deduped;

num_affected_rows,num_inserted_rows


In [0]:
-- Silver Step 5: Join community area names for readable output
CREATE OR REPLACE TABLE silver.chicago_crimes_clean
USING DELTA AS
SELECT
  c.*,  -- Keep all crime columns
  -- Add community area name in title case (e.g., "Near North Side")
  INITCAP(ca.community)                     AS community_area_name
FROM silver.crimes_enriched c
-- Left join to preserve all crime records even if area name missing
LEFT JOIN bronze.community_areas_raw ca
  -- Join on community area number (cast to string to match types)
  ON CAST(c.community_area AS STRING) = ca.area_numbe;

num_affected_rows,num_inserted_rows


In [0]:
-- Data Quality Check: Calculate null percentages for critical columns
SELECT
  -- For each column, count nulls and divide by total rows, then multiply by 100 for percentage
  ROUND(SUM(CASE WHEN crime_id            IS NULL THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) AS crime_id_null_pct,
  ROUND(SUM(CASE WHEN crime_type          IS NULL THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) AS crime_type_null_pct,
  ROUND(SUM(CASE WHEN date                IS NULL THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) AS date_null_pct,
  ROUND(SUM(CASE WHEN latitude            IS NULL THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) AS latitude_null_pct,
  ROUND(SUM(CASE WHEN community_area      IS NULL THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) AS community_area_null_pct,
  ROUND(SUM(CASE WHEN community_area_name IS NULL THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) AS area_name_null_pct,
  ROUND(SUM(CASE WHEN arrest              IS NULL THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) AS arrest_null_pct
FROM silver.chicago_crimes_clean;

crime_id_null_pct,crime_type_null_pct,date_null_pct,latitude_null_pct,community_area_null_pct,area_name_null_pct,arrest_null_pct
0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [0]:
-- Validation: Check for duplicate crime_ids after deduplication
-- Result should be 0 if deduplication worked correctly
SELECT COUNT(*) - COUNT(DISTINCT crime_id) AS duplicate_crime_ids
FROM silver.chicago_crimes_clean;

duplicate_crime_ids
0


In [0]:
-- Validation: Check data coverage (year range and record count)
SELECT
  MIN(year)  AS earliest_year,      -- First year in dataset
  MAX(year)  AS latest_year,        -- Last year in dataset
  MAX(date)  AS most_recent_record, -- Most recent crime record timestamp
  COUNT(*)   AS total_rows          -- Total number of records
FROM silver.chicago_crimes_clean;

earliest_year,latest_year,most_recent_record,total_rows
2019,2022,2022-12-31T23:55:00.000Z,904780


In [0]:
-- Gold Layer: Create aggregated analytics table by community area and year
CREATE OR REPLACE TABLE gold.mart_crime_by_area
USING DELTA AS
SELECT
  community_area,
  community_area_name,
  year,
  -- Count total crimes per area per year
  COUNT(*)                                        AS total_crimes,
  -- Count crimes that resulted in arrest
  SUM(CASE WHEN arrest = true THEN 1 ELSE 0 END)  AS total_arrests,
  -- Calculate arrest rate as percentage
  ROUND(
    SUM(CASE WHEN arrest = true THEN 1 ELSE 0 END)
    / COUNT(*) * 100, 2
  )                                               AS arrest_rate_pct,
  -- Rank areas by crime count (highest first) within each year
  RANK() OVER (
    PARTITION BY year
    ORDER BY COUNT(*) DESC
  )                                               AS crime_rank_desc,
  -- Rank areas by crime count (lowest first) within each year
  RANK() OVER (
    PARTITION BY year
    ORDER BY COUNT(*) ASC
  )                                               AS crime_rank_asc
FROM silver.chicago_crimes_clean
-- Group by area and year to get aggregated statistics
GROUP BY community_area, community_area_name, year;

num_affected_rows,num_inserted_rows
